In [1]:
using Lux, LuxCUDA, MLUtils
using Optimisers, Random, Statistics
using Zygote
using DiffEqFlux, OrdinaryDiffEq

In [2]:
using Images, JLD2
using ComponentArrays
using DeepEquilibriumNetworks
using Plots
using Dates

In [3]:
const cdev = cpu_device()
const gdev = gpu_device()
dev = gdev

(::CUDADevice{Nothing, Missing}) (generic function with 1 method)

In [ ]:
include("model.jl")

In [ ]:
rng = Xoshiro(0)
ps, st = Lux.setup(rng, model)

In [ ]:
ps = ps |> ComponentArray |> dev
st = st |> dev

In [ ]:
opt = Optimisers.NAdam(1e-4)
state = Optimisers.setup(opt,ps)
train_state = Lux.Training.TrainState(model, ps, st, opt)

In [ ]:
function loss_function(model, ps, st, (x, y_true))
    y_pred = model(x, ps, st)[1][1]
    loss_mse= MSELoss()
    mse_loss = loss_mse(y_pred, y_true)
    #sptrl_loss = loss_mse(dct(dct(y_pred,1),2),dct(dct(y_pred,1),2))
    return mse_loss, st
    #return mes_loss + sptrl_loss, st
end

In [ ]:
x = randn(Float32,128,128,1,4)
y_true = randn(Float32,128,128,1,4)
x_dev = x |> dev
y_dev = y_true |> dev
y, _ = model(x_dev, ps, st)